# Pulling monthly cost, driving, and premium data

Pulls five series from FRED and the motor vehicle insurance index from BLS, cleans them, and saves them to a SQLite database

In [1]:
import os
import requests
import pandas as pd
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv("FRED_API_KEY")
bls_key = os.getenv("BLS_API_KEY")

Five series come from FRED: four CPI series that drive claim severity (repair, parts, used cars, medical care) and vehicle miles traveled as an exposure proxy. All are seasonally adjusted.

In [2]:
#creating our loop
#fred_series = {} pairs each code with a readable name
#frames = [] is an empty list; each series' table gets added one by one
#for code, name in fred_series.items() goes through the fred_series dictionary one pair at a time
#series_id... tells FRED what data set I want, my key, and what format (json)
#requests.get() and pd.Dataframe() pull date and value columns
#df["series"] = name adds a column that labels every row with the series name, like "repair".
#frames.append(df) adds this round's table to the list
#pd.concat(frames) stacks all five tables into one large table
#this loop is essentially our test cell repeated five times, swapping the code and name each round

fred_series = {"CUSR0000SETD": "repair",
               "CUSR0000SETC": "parts",
               "CUSR0000SETA02": "used_cars",
               "CPIMEDSL": "medical",
               "TRFVOLUSM227SFWA": "miles_traveled"}

url = "https://api.stlouisfed.org/fred/series/observations"
frames = []

for code, name in fred_series.items():
    params = {"series_id": code, "api_key": api_key, "file_type": "json"}
    response = requests.get(url, params=params)
    df = pd.DataFrame(response.json()["observations"])[["date", "value"]]
    df["series"] = name
    frames.append(df)

fred = pd.concat(frames, ignore_index=True)
fred

,date,value,series
0,1967-01-01,29.800,repair
1,1967-02-01,29.900,repair
2,1967-03-01,30.000,repair
3,1967-04-01,30.100,repair
4,1967-05-01,30.200,repair
...,...,...,...
3307,2026-04-01,278673.0,miles_traveled
3308,2026-05-01,277066.0,miles_traveled
3309,2026-06-01,277987.0,miles_traveled
3310,2026-07-01,278172.0,miles_traveled


In [3]:
#check the list of names
fred["series"].unique()

<StringArray>
['repair', 'parts', 'used_cars', 'medical', 'miles_traveled']
Length: 5, dtype: str

In [4]:
#check the series came through: 3312 rows, 3 columns
fred.shape

(3312, 3)

In [5]:
#convert date and value into date and numbers, respectively, from text
fred["date"] = pd.to_datetime(fred["date"])
fred["value"] = pd.to_numeric(fred["value"], errors="coerce")

fred[fred["value"].isna()]

,date,value,series
705,2025-10-01,NaN,repair
1135,2025-10-01,NaN,parts
2979,2025-10-01,NaN,medical


October 2025 is missing for repair, parts, and medical because BLS didn't collect CPI prices during the government shutdown. Used cars wasn't affected since BLS builds it from dealer sales data, and miles traveled comes from the Department of Transportation.


FRED doesn't carry the motor vehicle insurance price index, the most important series in this project, so I pulled it directly from the U.S. Bureau of Labor Statistics (BLS), which creates the CPI. The other five series come from FRED because its API is simpler to use, and miles traveled comes from the Department of Transportation, not BLS.

In [6]:
#pull the data from BLS

#build the request
bls_url = "https://api.bls.gov/publicAPI/v2/timeseries/data/"
payload = {"seriesid": ["CUSR0000SETE"],
           "startyear": "2010",
           "endyear": "2026",
           "registrationkey": bls_key}

#send and receive
bls_response = requests.post(bls_url, json=payload)
bls_json = bls_response.json()
print(bls_json["status"])

#turn the reply into a table
insurance = pd.DataFrame(bls_json["Results"]["series"][0]["data"])

#match the FRED format, organize our data!
insurance = insurance[insurance["period"] != "M13"]
insurance["date"] = pd.to_datetime(insurance["year"] + "-" + insurance["period"].str[1:] + "-01")
insurance["value"] = pd.to_numeric(insurance["value"], errors="coerce")
insurance["series"] = "insurance"
insurance = insurance[["date", "value", "series"]].sort_values("date")
insurance.head()

REQUEST_SUCCEEDED


,date,value,series
200,2010-01-01,366.349,insurance
199,2010-02-01,368.421,insurance
198,2010-03-01,370.678,insurance
197,2010-04-01,372.452,insurance
196,2010-05-01,373.569,insurance


In [7]:
#check for missing months, whether blank or left out entirely
all_months = pd.date_range(insurance["date"].min(), insurance["date"].max(), freq="MS")
all_months.difference(insurance.dropna(subset=["value"])["date"])

DatetimeIndex(['2025-10-01', '2025-11-01'], dtype='datetime64[us]', freq='MS')

Insurance is missing two months: October 2025 comes through as a blank, and November 2025 isn't in the BLS data at all, so it only shows up when checking for gaps in the dates.

I stacked the FRED and BLS tables into one and saved it to a SQLite database. The table is replaced on every run because BLS revises seasonally adjusted values each February.

In [8]:
import sqlite3

#stack the FRED and BLS tables into one
trend = pd.concat([fred, insurance], ignore_index=True)
trend["date"] = trend["date"].dt.strftime("%Y-%m-%d")

#save to the database (creates the file if it doesn't exist)
conn = sqlite3.connect("../data/trend.db")
trend.to_sql("monthly_series", conn, if_exists="replace", index=False)

3513

In [9]:
#check each series made it into the database
#first SQL query: triple quotes hold the multi-line SQL text, and pd.read_sql runs it
check = """
SELECT series,
       COUNT(*) AS months,
       MIN(date) AS first_month,
       MAX(date) AS last_month
FROM monthly_series
GROUP BY series
"""
result = pd.read_sql(check, conn)
conn.close()
result

The data has been pulled, cleaned, checked, and stored in SQL.